# Tarea 5: Extracción de características

> Uriel Aguilar Mauricio

***Librerias***

In [40]:
import ast
import pandas as pd
from radon.complexity import cc_visit

***Código para el ejercicio***

In [41]:
codigo = """class Cuenta:

    def depositar(self, monto):
        if monto > 0:
            self.saldo += monto

    def retirar(self, monto):
        if monto < self.saldo:
            self.saldo -= monto
        else:
            print("Fondos insuficientes")
"""

***Contar las lineas de código***

In [42]:
def n_loc(codigo: str) -> int:
    lineas = codigo.splitlines()

    return sum(
        1
        for linea in lineas
        if linea.strip() and not linea.strip().startswith("#")
    )

***Ocupar ast para obtener el árbol sintáctico***

In [43]:
def obtener_ast(codigo: str) -> dict:
    arbol = ast.parse(codigo)

    clases = sum(
        isinstance(nodo, ast.ClassDef)
        for nodo in ast.walk(arbol)
    )

    funciones = sum(
        isinstance(nodo, (ast.FunctionDef, ast.AsyncFunctionDef))
        for nodo in ast.walk(arbol)
    )

    decisiones_if = sum(
        isinstance(nodo, ast.If)
        for nodo in ast.walk(arbol)
    )

    return {
        "Clases": clases,
        "Funciones": funciones,
        "Decisiones_if": decisiones_if
    }

***Complejidad ciclomatica con radon***

In [44]:
def obtener_complejidad(codigo: str) -> list:
    resultados = cc_visit(codigo)

    detalle = []

    for elemento in resultados:
        detalle.append({
            "Nombre": elemento.name,
            "Tipo": elemento.__class__.__name__,
            "Complejidad": elemento.complexity
        })

        if hasattr(elemento, "methods"):
            for metodo in elemento.methods:
                detalle.append({
                    "Nombre": f"{elemento.name}.{metodo.name}",
                    "Tipo": "Method",
                    "Complejidad": metodo.complexity
                })

    return detalle

***Extraer caracterisiticas***

In [45]:
def extraer_caracteristicas(codigo: str) -> dict:
    metricas_ast = obtener_ast(codigo)
    complejidad = obtener_complejidad(codigo)

    complejidades = [
        elemento["Complejidad"]
        for elemento in complejidad
        if elemento["Tipo"] != "Class"
    ]

    return {
        "LOC": n_loc(codigo),
        "Clases": metricas_ast["Clases"],
        "Funciones": metricas_ast["Funciones"],
        "Decisiones_if": metricas_ast["Decisiones_if"],
        "Complejidad_total": sum(complejidades),
        "Complejidad_maxima": max(complejidades, default=0)
    }

***Ver caracteristicas***

In [46]:
metricas = extraer_caracteristicas(codigo)

pd.DataFrame([metricas])

,LOC,Clases,Funciones,Decisiones_if,Complejidad_total,Complejidad_maxima
0,9,1,2,2,8,2


***Ver complejidad ciclomatica***

In [47]:
detalle_complejidad = pd.DataFrame(obtener_complejidad(codigo))
detalle_complejidad


,Nombre,Tipo,Complejidad
0,Cuenta,Class,3
1,Cuenta.depositar,Method,2
2,Cuenta.retirar,Method,2
3,depositar,Function,2
4,retirar,Function,2


## 11. Conclusión

La extracción automática de características permite transformar código fuente en datos cuantitativos que pueden utilizarse para analizar atributos relacionados con tamaño y complejidad del software.

En este ejercicio:

- `LOC` aproxima el tamaño del código.
- El número de clases y funciones describe parte de su estructura.
- El número de decisiones muestra la presencia de lógica condicional.
- La complejidad ciclomática cuantifica la cantidad de caminos independientes del flujo de control.
